In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("GEMINI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"GEMINI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("GEMINI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'GEMINI_API_KEY', and load_dotenv() ran without error.")

GEMINI_API_KEY loaded (53 characters): AQ.A...EMNg


In [3]:
!pip install --upgrade google-generativeai


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: c:\Users\olagunju\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip



INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
  Using cached httplib2-0.32.0-py3-none-any.whl.metadata (2.2 kB)
INFO: pip is looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of proto-plus to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: Thi

In [12]:
"""
Few-shot relevance classification using Gemini (google.genai SDK).

Same crash-safe / resumable design as the other classify_with_* scripts:
writes each result immediately, skips already-processed wos_ids on rerun.

Requires: pip install google-genai python-dotenv
Requires a .env file (not committed, not shared) with:
    GEMINI_API_KEY=...

Model names updated to the current generation (as of Aug 2026); check
https://ai.google.dev/gemini-api/docs/models if these have moved on by the
time you run this.
"""

import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found. Add it to your .env file.")

# ---- Config ----
MODEL_NAME = "gemini-3.6-flash"  # "Pro" tier; swap to "gemini-3.6-flash"
                                          # for the cheaper/faster comparison run
PROMPT_FILE = "gpt_classification_prompt.txt"
INPUT_FILE = "merged_shuffled.xlsx"        # columns: wos_id, title, abstract
OUTPUT_JSONL = "classification_results_gemini-flash.jsonl"

# Retry settings for transient 503 "model overloaded" errors -- these showed
# up repeatedly on gemini-3.1-pro-preview in prior runs, so backoff is real,
# not precautionary.
MAX_RETRIES = 4
RETRY_BACKOFF_SEC = 15  # doubles each retry: 15, 30, 60, 120


def load_system_prompt(path: str) -> str:
    return Path(path).read_text().strip()


def load_input_papers(path: str) -> pd.DataFrame:
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


def already_processed_ids(output_path: str) -> set:
    if not os.path.exists(output_path):
        return set()
    ids = set()
    with open(output_path) as f:
        for line in f:
            try:
                ids.add(json.loads(line)["wos_id"])
            except (json.JSONDecodeError, KeyError):
                continue
    return ids


def extract_json(text: str) -> dict | None:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.MULTILINE)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def build_user_message(title: str, abstract: str) -> str:
    return f"Title: {title}\nAbstract: {abstract}"


def call_with_retries(client, system_prompt: str, user_message: str) -> str:
    attempt = 0
    while True:
        attempt += 1
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=user_message,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    temperature=0,
                ),
            )
            return response.text
        except Exception as e:
            is_503 = "503" in str(e) or "UNAVAILABLE" in str(e)
            if is_503 and attempt <= MAX_RETRIES:
                wait = RETRY_BACKOFF_SEC * (2 ** (attempt - 1))
                print(f"  Overloaded, retry {attempt}/{MAX_RETRIES} in {wait}s...")
                time.sleep(wait)
                continue
            raise


def main():
    client = genai.Client(api_key=GEMINI_API_KEY)
    system_prompt = load_system_prompt(PROMPT_FILE)
    papers = load_input_papers(INPUT_FILE)
    done_ids = already_processed_ids(OUTPUT_JSONL)
    print(f"Model: {MODEL_NAME}")
    print(f"Total papers: {len(papers)} | Already processed: {len(done_ids)}")

    n_ok, n_parse_failed, n_api_failed = 0, 0, 0
    flagged = []

    with open(OUTPUT_JSONL, "a") as out_f:
        for row in papers.itertuples():
            wos_id = row.wos_id
            if wos_id in done_ids:
                continue

            user_message = build_user_message(row.title, row.abstract)
            try:
                generated = call_with_retries(client, system_prompt, user_message)
            except Exception as e:
                print(f"  FAILED for {wos_id}: {e}")
                n_api_failed += 1
                flagged.append(wos_id)
                continue

            parsed = extract_json(generated)
            result = {"wos_id": wos_id, "title": row.title, "raw_model_output": generated}
            if parsed is not None and "label" in parsed:
                result.update(parsed)
                n_ok += 1
            else:
                result.update({"label": None, "trap_reason": None, "reason": None, "parse_failed": True})
                n_parse_failed += 1
                flagged.append(wos_id)

            out_f.write(json.dumps(result) + "\n")
            out_f.flush()

    print(f"\nDone. OK: {n_ok} | Parse failed: {n_parse_failed} | API failed: {n_api_failed}")
    if flagged:
        print(f"Flagged wos_ids: {flagged[:20]}{' ...' if len(flagged) > 20 else ''}")
    print(f"Results saved to {OUTPUT_JSONL}")


if __name__ == "__main__":
    main()

Model: gemini-3.6-flash
Total papers: 433 | Already processed: 0


  Overloaded, retry 1/4 in 15s...


  Overloaded, retry 1/4 in 15s...


  Overloaded, retry 1/4 in 15s...



Done. OK: 433 | Parse failed: 0 | API failed: 0
Results saved to classification_results_gemini-flash.jsonl


In [13]:
"""
Convert a classification results .jsonl file (from classify_with_open_llm.py,
or your dpo_pairs.jsonl / train_pairs.jsonl / validation_pairs.jsonl) into an
.xlsx file for easy review in Excel.

Usage: edit INPUT_PATH and OUTPUT_PATH below, then run.
"""

import json
import pandas as pd

INPUT_PATH = "classification_results_gemini-flash.jsonl"
OUTPUT_PATH = "LLM-FULL/classification_results_gemini-flash.xlsx"


def main():
    rows = []
    with open(INPUT_PATH) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    df = pd.DataFrame(rows)
    df.to_excel(OUTPUT_PATH, index=False)
    print(f"Converted {len(rows)} rows from {INPUT_PATH} -> {OUTPUT_PATH}")
    print(f"Columns: {list(df.columns)}")


if __name__ == "__main__":
    main()

Converted 433 rows from classification_results_gemini-flash.jsonl -> LLM-FULL/classification_results_gemini-flash.xlsx
Columns: ['wos_id', 'title', 'raw_model_output', 'label', 'trap_reason', 'reason']


In [9]:
"""
Convert a classification results .jsonl file (from classify_with_open_llm.py,
or your dpo_pairs.jsonl / train_pairs.jsonl / validation_pairs.jsonl) into an
.xlsx file for easy review in Excel.

Usage: edit INPUT_PATH and OUTPUT_PATH below, then run.
"""

import json
import pandas as pd

INPUT_PATH = "classification_results_gemini.jsonl"
OUTPUT_PATH = "LLM-Result/classification_results_gemini.xlsx"


def main():
    rows = []
    with open(INPUT_PATH) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    df = pd.DataFrame(rows)
    df.to_excel(OUTPUT_PATH, index=False)
    print(f"Converted {len(rows)} rows from {INPUT_PATH} -> {OUTPUT_PATH}")
    print(f"Columns: {list(df.columns)}")


if __name__ == "__main__":
    main()

Converted 245 rows from classification_results_gemini.jsonl -> LLM-Result/classification_results_gemini.xlsx
Columns: ['wos_id', 'title', 'raw_model_output', 'label', 'trap_reason', 'reason']


In [ ]:
"""
Re-parse rows that failed JSON extraction, using a hardened parser that
handles two common, harmless LLM formatting quirks:
  1. Literal newline characters inside a JSON string value (should be
     escaped as \\n, but the model sometimes just line-wraps the text).
  2. Curly/smart quotes (" " ' ') used instead of straight ASCII quotes.

This does NOT call the model again -- it re-parses raw_model_output, which
is already saved from the original run. Only rows where parsing genuinely
fixes something are updated; rows still unparseable (e.g. the model
abandoned JSON entirely) stay flagged for manual review.

Usage: edit INPUT_PATH and OUTPUT_PATH, then run.
"""

import json
import re
import pandas as pd

INPUT_PATH = "LLM-FULL/classification_results_gemini.xlsx"
OUTPUT_PATH = "LLM-FULL/classification_results_gemini_reparsed.xlsx"


def hardened_extract_json(text: str) -> dict | None:
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    candidate = match.group(0)

    # Fix 1: normalize smart/curly quotes to straight ASCII quotes.
    candidate = (
        candidate.replace("\u201c", '"').replace("\u201d", '"')
        .replace("\u2018", "'").replace("\u2019", "'")
    )

    # Try parsing as-is first.
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Fix 2: escape literal newlines/tabs that fall INSIDE string values.
    # A simple, safe approach for this use case: replace raw newlines/tabs
    # with a space, since these failures are all mid-sentence line wraps,
    # not intentional formatting.
    candidate_fixed = candidate.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    candidate_fixed = re.sub(r"\s+", " ", candidate_fixed)
    try:
        return json.loads(candidate_fixed)
    except json.JSONDecodeError:
        return None


def main():
    df = pd.read_excel(INPUT_PATH) if INPUT_PATH.endswith((".xlsx", ".xls")) else pd.read_csv(INPUT_PATH)

    if "parse_failed" not in df.columns:
        print("No 'parse_failed' column found -- nothing to re-parse.")
        return

    failed_mask = df["parse_failed"] == True
    n_failed_before = failed_mask.sum()
    print(f"Rows previously flagged as parse_failed: {n_failed_before}")

    # Force flexible dtype on columns we're about to write mixed values into,
    # since pandas can otherwise reject e.g. writing a bool into a float64
    # column that only had True/NaN in it originally.
    for col in ["label", "trap_reason", "reason", "parse_failed"]:
        if col in df.columns:
            df[col] = df[col].astype(object)

    n_recovered = 0
    still_failed = []

    for idx in df[failed_mask].index:
        raw = df.at[idx, "raw_model_output"]
        parsed = hardened_extract_json(raw)
        if parsed is not None and "label" in parsed:
            df.at[idx, "label"] = parsed.get("label")
            df.at[idx, "trap_reason"] = parsed.get("trap_reason")
            df.at[idx, "reason"] = parsed.get("reason")
            df.at[idx, "parse_failed"] = False
            n_recovered += 1
        else:
            still_failed.append(df.at[idx, "wos_id"])

    df.to_excel(OUTPUT_PATH, index=False)

    print(f"Recovered: {n_recovered}")
    print(f"Still failed (genuine issue, needs manual review or a rerun "
          f"for just this row): {len(still_failed)}")
    if still_failed:
        print(f"  -> {still_failed}")
    print(f"Saved to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

No 'parse_failed' column found -- nothing to re-parse.
